<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

# Network Names and Service Discovery

A Duckiedrone's IP address can change when it joins another network or receives a new DHCP lease. A hostname, a discovered device, a reachable address, and a working service are different pieces of evidence. This notebook explains how a hostname and a Uniform Resource Locator (URL) fit into that chain.

## Local name resolution

DNS maps names to IP addresses. Ordinary DNS normally asks a configured DNS server. On a local network, mDNS can resolve names ending in `.local` without a central DNS server. `DUCKIEDRONE_NAME.local` can therefore be easier to remember than a changing DHCP address.

A simplified local mDNS question and response are shown in [Figure 1](#figure-1).

<figure id="figure-1" style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    base station queries amelia.local
      |
      v
    local mDNS query
      |
      v
    amelia.local returns 192.168.1.201
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">Figure 1: A local mDNS lookup for <code>amelia.local</code>.</figcaption>
</figure>

Names ending in `.local` normally use mDNS rather than ordinary DNS. They are meaningful only on the local network unless that network deliberately provides an mDNS relay or proxy. `localhost` is different: it names the loopback address of the computer running the command, not another computer.

mDNS uses multicast traffic that nearby devices can receive together. Guest networks, enterprise Wi-Fi, and separate virtual local area networks (VLANs) can block it, so a Duckiedrone can be powered on while `DUCKIEDRONE_NAME.local` fails to resolve.

### Read a lookup result

On an Ubuntu base station connected to the intended robot network, a lookup can return an address such as:

```shell
192.168.1.201    amelia.local
```

The first field is the returned address. A lookup can instead return an IPv6 address, several lines, or no answer. No answer establishes only that lookup failed in the current environment; it does not prove that the Duckiedrone is powered off.

## Web addresses and trust

A URL can include a protocol, hostname, optional port, and path. Hypertext Transfer Protocol Secure (HTTPS) is the encrypted form of HTTP. In `https://DUCKIEDRONE_NAME.local:8443/status`, `https` selects HTTPS, `DUCKIEDRONE_NAME.local` is the host, `8443` is the port, and `/status` is the resource. The port is optional when the protocol has a standard default.

The components of this example URL are summarized in [Table 1](#table-1).

<table id="table-1" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 1: Components of the example URL.</caption>
  <thead>
    <tr>
      <th>Component</th>
      <th>Meaning</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>https</code></td>
      <td>Use HTTPS, the encrypted form of HTTP</td>
    </tr>
    <tr>
      <td><code>DUCKIEDRONE_NAME.local</code></td>
      <td>Resolve this hostname to an address</td>
    </tr>
    <tr>
      <td><code>8443</code></td>
      <td>Contact the service on this port</td>
    </tr>
    <tr>
      <td><code>/status</code></td>
      <td>Request this resource from the service</td>
    </tr>
  </tbody>
</table>

If a port is omitted, the browser uses the scheme's default: `80` for HTTP or `443` for HTTPS. Changing `http` to `https` does not enable HTTPS on a service.

Opening this URL in a browser depends on resolving its hostname, finding a route to the target device, and connecting to the correct TCP port. A browser certificate warning means the browser contacted a server but cannot yet establish trust in its certificate. Verify the expected address and service rather than accepting a warning blindly.

## Separate the connection questions

- __Discovery__ asks whether local announcements or a discovery check can see a compatible Duckiedrone.

- __Name resolution__ asks whether a known name such as `DUCKIEDRONE_NAME.local` maps to an address.

- __Reachability__ asks whether packets can travel to that address. `ping` provides one form of evidence, but a firewall can block it.

- __Service availability__ asks whether a particular protocol and port accept a connection.

A successful hostname lookup completes only name resolution; the route, reachability, and service need separate checks. A missing discovery result does not prove that a Duckiedrone is unavailable.

The evidence each connection question provides is distinguished in [Table 2](#table-2).

<table id="table-2" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 2: Evidence from discovery, name resolution, reachability, and service checks.</caption>
  <thead>
    <tr>
      <th>Question</th>
      <th>Example evidence</th>
      <th>What remains unproven</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Discovery</td>
      <td><code>dts fleet discover</code> lists a Duckiedrone</td>
      <td>Whether its dashboard responds</td>
    </tr>
    <tr>
      <td>Name resolution</td>
      <td><code>getent</code> returns an address</td>
      <td>Whether that address is reachable</td>
    </tr>
    <tr>
      <td>Reachability</td>
      <td>A Duckiedrone replies to <code>ping</code></td>
      <td>Whether a particular service accepts connections</td>
    </tr>
    <tr>
      <td>Service availability</td>
      <td>The dashboard returns an HTTP response</td>
      <td>Whether the requested operation succeeds</td>
    </tr>
  </tbody>
</table>

`dts fleet discover` is a targeted compatible-device discovery check. It looks for compatible Duckiedrones through local announcements and discovery responses, rather than scanning addresses, ports, or unfamiliar devices. It refreshes a table that can show a hostname, hardware, model, and a status such as `Booting` or `Ready`. It combines local mDNS announcements with a UDP discovery check, so guest networks, device-group separation, multicast policy, or UDP policy can prevent an entry from appearing. Press `Ctrl-C` to stop it. A missing result is evidence to record and compare with the device and network documentation; it is not permission to broaden the search.

### Try it

From the base-station terminal, first resolve the local host:

```bash
getent hosts localhost
```

Only for a Duckiedrone and network you are authorized to inspect, continue with:

```bash
getent hosts DUCKIEDRONE_NAME.local
dts fleet discover
```

Record whether each command provides name-resolution or discovery evidence. Do not treat an absent result as permission to scan addresses, ports, or unfamiliar devices.

<details>
<summary>Check your result</summary>

`getent hosts` asks the configured name-resolution sources about one name. `dts fleet discover` looks for compatible local announcements and discovery responses. Neither command proves that a particular service accepts a connection.

</details>

## Further reading

The IETF [mDNS specification](https://www.rfc-editor.org/rfc/rfc6762) describes local `.local` name resolution.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
